# UMNA benchmark results

This notebook loads the published result files and plots them. It does
not run UMNA: the source is not published yet. Every number here comes
from a file in `results/`, whose hash is listed in PROVENANCE.md.

Run the cells in order (Runtime > Run all).

In [ ]:
import json, os, urllib.request
import matplotlib.pyplot as plt

REPO, BRANCH = "nabvian/umna-benchmarks", "main"
LOCAL = next((p for p in ("../results", "results") if os.path.isdir(p)), None)

def load(name):
    """A result file, from a local checkout if there is one, else from GitHub."""
    if LOCAL:
        with open(os.path.join(LOCAL, name)) as f:
            return json.load(f)
    url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/results/{name}"
    with urllib.request.urlopen(url) as f:
        return json.load(f)

print("reading from", LOCAL or f"github.com/{REPO}")

## Where each claim stands after five more seeds

A claim replicates if its own preregistered verdict holds on all five.

In [ ]:
ms = load("multiseed_results.json")
names = {"c2": "H3  harmful updates", "capability_routing": "HC  capability routing",
         "drift": "HD  drift on the frozen Core", "n2": "H1  NICE-R vs greedy",
         "slow_harm_fine": "H3-slow  slow erosion (before gate minimum)"}
for test, row in ms["tests"].items():
    print(f"{names[test]:46} {row['supported']} of 5   {row['rule']}")
gp = load("gate_power_results.json")
print(f"\nWith the gate's minimum rows, eroded versions never served on six seeds: "
      f"{gp['verdict']['PG1 cce never serves an eroded version']}")

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh([names[t] for t in ms["tests"]], [r["supported"] for r in ms["tests"].values()], color="#4c72b0")
ax.set_xlim(0, 5); ax.set_xlabel("replicates supported (of 5)"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## Slow erosion: one more poisoned label per version

Audit accuracy of the version being served as 552 (spambase) or 126
(qsar-biodeg) versions are offered in turn. With the cumulative budget
switched off, the step test alone lets accuracy slide; CCE stops it
before the budget E (dashed line). Without any gate every version is
served, so that line lies under the step-only one.

In [ ]:
sh = load("slow_harm_fine_results.json")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, name in zip(axes, ("spambase", "qsar-biodeg")):
    e = sh["datasets"][name]
    origin = e["origin_audit_accuracy"]
    for arm, label in (("no_gate", "no gate"), ("cce_step_only", "step test only"), ("cce", "CCE")):
        served, current = [origin], origin
        for d in e["arms"][arm]["decisions"]:
            if d["promoted"]:
                current = d["audit_accuracy"]
            served.append(current)
        ax.plot(served, label=label)
    ax.axhline(origin - 0.05, ls="--", c="grey", lw=1)
    ax.set_title(name); ax.set_xlabel("version offered"); ax.set_ylabel("audit accuracy served")
axes[0].legend(); plt.tight_layout(); plt.show()

## Adding tasks: frozen Core against a shared network

Accuracy on each old task after every new task is added. UMNA's numbers
count a declined row as wrong; the section below scores both at the same
coverage.

In [ ]:
dr = load("drift_frozen_core_results.json")
stages = dr["stages"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for task in stages[-1]["tasks"]["no_cce"]:
    xs = [s["stage"] for s in stages if task in s["tasks"]["no_cce"]]
    axes[0].plot(xs, [s["tasks"]["no_cce"][task]["audit_served_accuracy"] for s in stages if task in s["tasks"]["no_cce"]], marker="o", label=task)
for task in stages[-1]["shared"]:
    xs = [s["stage"] for s in stages if task in s["shared"]]
    axes[1].plot(xs, [s["shared"][task]["audit_accuracy"] for s in stages if task in s["shared"]], marker="o", label=task)
axes[0].set_title("UMNA (frozen Core)"); axes[1].set_title("shared network, fine-tuned")
for ax in axes:
    ax.set_xlabel("tasks added")
axes[0].set_ylabel("audit accuracy"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Accuracy at the same coverage

Shared network minus UMNA, over six seeds, with 95% intervals. Below zero
means UMNA is more accurate.

In [ ]:
mc = load("matched_coverage_results.json")["verdict"]
labels = list(mc)
fig, ax = plt.subplots(figsize=(8, 3))
for i, k in enumerate(labels):
    v = mc[k]
    ax.errorbar(v["mean"], i, xerr=[[v["mean"] - v["low"]], [v["high"] - v["mean"]]], fmt="o", capsize=4)
ax.axvline(0, c="grey", lw=1); ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("shared minus UMNA"); plt.tight_layout(); plt.show()

## Capability routing

Net correct rate (right minus wrong, over all rows) for each task and
budget. NICE-R is not told which Wing serves which task; the gate routers
are normal learned routers.

In [ ]:
cr = load("capability_routing_results_fixed_estimator.json")["base"]
arms = ["nice_r", "gate_top1", "gate_top2", "greedy"]
cells = [(t, b) for t in cr for b in cr[t]["cells"]]
fig, ax = plt.subplots(figsize=(11, 3.4))
width = 0.2
for j, arm in enumerate(arms):
    ax.bar([i + j * width for i in range(len(cells))], [cr[t]["cells"][b][arm]["net_correct_rate"] for t, b in cells], width, label=arm)
ax.set_xticks([i + 1.5 * width for i in range(len(cells))]); ax.set_xticklabels([f"{t}\n{b}" for t, b in cells], fontsize=8)
ax.set_ylabel("net correct rate"); ax.legend(ncol=4, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0)); plt.tight_layout(); plt.show()

## The gate's minimum rows

Gate rows each dataset has, against the minimum a non-inferiority test at
the budget needs, for each seed. Below the line the gate refuses to
decide.

In [ ]:
seeds = gp["seeds"]
fig, ax = plt.subplots(figsize=(7, 3))
for name in ("spambase", "banknote-authentication", "qsar-biodeg"):
    line, = ax.plot(list(seeds), [seeds[s]["slow_harm"][name]["minimum"] for s in seeds], marker="o", label=f"{name}: minimum")
    ax.axhline(seeds["0"]["slow_harm"][name]["gate_rows"], ls=":", c=line.get_color(), label=f"{name}: rows available")
ax.set_xlabel("seed"); ax.set_ylabel("rows"); ax.set_yscale("log"); ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
plt.tight_layout(); plt.show()